# H5 — Threat Imminence Gradient (THE CORE HYPOTHESIS)

**Prediction:** Effort-threat integration occurs specifically in the pre-encounter deliberative phase and dissolves at predator encounter, when a separate reactive defense system takes over.

**Sub-hypotheses:**
- **H5a:** Threat x beta interaction on residualized vigor: significant in anticipatory (p < 0.05), null in reactive (p > 0.10)
- **H5b:** Distance x c_d interaction: null in anticipatory (p > 0.10), significant in reactive (p < 0.01)
- **H5c:** c_d main effect: null in anticipatory (p > 0.10), significant in terminal (p < 0.001)
- **H5d:** Threat predicts vigor in anticipatory (p < 0.01) but not terminal (p > 0.10)
- **H5e:** Reactive vigor driven by actual predator presence (attack_trial), not stated probability; interaction null
- **H5f:** Between-subject variance compressed in reactive vs anticipatory (ratio > 3; exploratory: 6.6x)

**What this determines:** Whether the threat imminence continuum is reflected in within-trial vigor dynamics — deliberative integration pre-encounter dissolving into a reflexive, parameter-independent defense response post-encounter.

In [ ]:
# ── Imports & Data Loading (self-contained) ──
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import ast
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import pearsonr, zscore
from pathlib import Path

%matplotlib inline
plt.rcParams.update({
    'figure.dpi': 120, 'font.size': 10,
    'axes.spines.right': False, 'axes.spines.top': False,
})

DATA_DIR = Path("/workspace/data/exploratory_350/processed/stage5_filtered_data_20260320_191950")
OUT_DIR  = Path("/workspace/results/stats/full_analysis")
EXCLUDE  = [154, 197, 208]

# ── Load pre-computed epoch data (with residualized vigor) ──
epoch_data = pd.read_csv(OUT_DIR / "part2_epoch_data.csv")
print(f"Epoch data: {len(epoch_data):,} rows, {epoch_data['subj'].nunique()} subjects")
print(f"Columns: {list(epoch_data.columns)}")

# ── Load params ──
params = pd.read_csv(OUT_DIR / "part1_params_full.csv")

# ── Also load raw behavior_rich for any recomputation needed ──
beh_rich = pd.read_csv(DATA_DIR / "behavior_rich.csv", low_memory=False)
beh_rich = beh_rich[~beh_rich['subj'].isin(EXCLUDE)].copy()
beh_rich['T_round'] = beh_rich['threat'].round(1)
beh_rich['actual_dist'] = beh_rich['startDistance'].map({5: 1, 7: 2, 9: 3})
beh_rich['is_heavy'] = (beh_rich['trialCookie_weight'] == 3.0).astype(int)
beh_rich['enc_t'] = pd.to_numeric(beh_rich['encounterTime'], errors='coerce')
beh_rich['strike_t'] = pd.to_numeric(beh_rich['strike_time'], errors='coerce')
beh_rich['is_attack'] = beh_rich['isAttackTrial'].astype(int)

print(f"All trials: {len(beh_rich):,}")

## Step 1: Compute epoch vigor and residualize

Vigor = median(1/IPI) / calibrationMax per epoch.  
Residualize each epoch on cookie_type + trial_number with random slopes by participant.

In [ ]:
# ── Step 1: Compute epoch vigor if not already in epoch_data ──
# Check if residualized columns exist
has_resids = all(c in epoch_data.columns for c in ['antic_vigor_resid', 'react_vigor_resid', 'term_vigor_resid'])

if has_resids:
    print("Residualized epoch vigor already present in epoch_data.")
    ed = epoch_data.copy()
else:
    print("Computing epoch vigor from raw data...")
    def compute_epoch_vigor_row(row):
        """Compute median(1/IPI)/calibrationMax for each epoch."""
        try:
            pt = np.array(ast.literal_eval(str(row['alignedEffortRate'])), dtype=float)
            if len(pt) < 5:
                return pd.Series({'antic_vigor': np.nan, 'react_vigor': np.nan, 'term_vigor': np.nan})
            enc = row['enc_t']
            strike = row['strike_t']
            cal = row['calibrationMax']
            if pd.isna(enc) or enc <= 0 or cal <= 0:
                return pd.Series({'antic_vigor': np.nan, 'react_vigor': np.nan, 'term_vigor': np.nan})

            def epoch_v(epoch_pts):
                if len(epoch_pts) < 4:
                    return np.nan
                ipis = np.diff(epoch_pts)
                ipis = ipis[ipis > 0.01]
                if len(ipis) < 3:
                    return np.nan
                return np.median((1.0 / ipis) / cal)

            antic = epoch_v(pt[pt < enc])
            react = epoch_v(pt[(pt >= enc) & (pt < enc + 2)])
            term = (epoch_v(pt[(pt >= strike - 2) & (pt <= strike)])
                    if row['is_attack'] == 1 and not pd.isna(strike) and strike > enc + 2
                    else np.nan)
            return pd.Series({'antic_vigor': antic, 'react_vigor': react, 'term_vigor': term})
        except:
            return pd.Series({'antic_vigor': np.nan, 'react_vigor': np.nan, 'term_vigor': np.nan})

    vigor_cols = beh_rich.apply(compute_epoch_vigor_row, axis=1)
    beh_rich = pd.concat([beh_rich, vigor_cols], axis=1)
    beh_rich['trial_number'] = beh_rich.groupby('subj').cumcount() + 1

    # Residualize each epoch
    def residualize(df, col):
        edf = df.dropna(subset=[col]).copy()
        edf['tn'] = edf.groupby('subj').cumcount() + 1
        try:
            m = smf.mixedlm(f"{col} ~ is_heavy + tn", edf, groups=edf["subj"],
                            re_formula="~is_heavy").fit(reml=False)
        except:
            m = smf.mixedlm(f"{col} ~ is_heavy + tn", edf, groups=edf["subj"]).fit(reml=False)
        edf[f'{col}_resid'] = m.resid
        return edf[['subj', 'trial', f'{col}_resid']]

    antic_r = residualize(beh_rich, 'antic_vigor')
    react_r = residualize(beh_rich, 'react_vigor')
    term_r  = residualize(beh_rich, 'term_vigor')

    ed = beh_rich[['subj', 'trial', 'T_round', 'actual_dist', 'is_heavy', 'is_attack']].copy()
    ed = ed.rename(columns={'actual_dist': 'distance', 'is_heavy': 'cookie_type'})
    ed['predator_probability'] = ed['T_round']
    ed = ed.merge(antic_r, on=['subj', 'trial'], how='left')
    ed = ed.merge(react_r, on=['subj', 'trial'], how='left')
    ed = ed.merge(term_r, on=['subj', 'trial'], how='left')
    ed = ed.merge(params[['subj', 'log_k_z', 'log_beta_z', 'log_cd_z']], on='subj')
    print(f"Computed epoch data: {len(ed):,} rows")

print(f"\nEpoch data shape: {ed.shape}")
print(f"Non-null counts:")
for col in ['antic_vigor_resid', 'react_vigor_resid', 'term_vigor_resid']:
    print(f"  {col}: {ed[col].notna().sum():,}")

## Step 2: Verify signal in residuals

Quick check that residualized vigor still contains meaningful signal from task conditions.

In [ ]:
# ── Step 2: Verify residual signal ──
print("Signal check: mean residualized vigor by threat level and epoch")
print("=" * 65)
for epoch, col in [('Anticipatory', 'antic_vigor_resid'),
                   ('Reactive',     'react_vigor_resid'),
                   ('Terminal',     'term_vigor_resid')]:
    sub = ed.dropna(subset=[col])
    means = sub.groupby('T_round')[col].mean()
    print(f"\n  {epoch}:")
    for t, v in means.items():
        print(f"    T={t}: mean resid = {v:.4f}")

# Quick visualization
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, (epoch, col) in zip(axes, [('Anticipatory', 'antic_vigor_resid'),
                                     ('Reactive', 'react_vigor_resid'),
                                     ('Terminal', 'term_vigor_resid')]):
    sub = ed.dropna(subset=[col])
    means = sub.groupby('T_round')[col].agg(['mean', 'sem'])
    ax.bar(means.index, means['mean'], yerr=means['sem'] * 1.96,
           color=['forestgreen', 'gold', 'firebrick'], edgecolor='white', capsize=3)
    ax.set_xlabel('Threat Probability')
    ax.set_ylabel('Residualized Vigor')
    ax.set_title(epoch)
    ax.axhline(0, color='gray', ls='--', alpha=0.5)
    ax.set_xticks([0.1, 0.5, 0.9])

plt.suptitle('Residualized Vigor by Threat Level and Epoch', y=1.02)
plt.tight_layout()
plt.show()

## H5a-c — The 3-epoch interaction model (the big table)

Run per epoch:  
`vigor_resid ~ predator_probability + distance + beta_z + k_z + cd_z + predator_probability:beta_z + predator_probability:k_z + predator_probability:cd_z + distance:beta_z + distance:k_z + distance:cd_z + (1 | participant)`

Key tests:
- H5a: threat x beta — sig anticipatory, null reactive
- H5b: distance x c_d — null anticipatory, sig reactive
- H5c: c_d main — null anticipatory, sig terminal

In [ ]:
# ── H5a-c: 3-epoch interaction models ──
formula = ("{dv} ~ predator_probability + distance + log_beta_z + log_k_z + log_cd_z + "
           "predator_probability:log_beta_z + predator_probability:log_k_z + predator_probability:log_cd_z + "
           "distance:log_beta_z + distance:log_k_z + distance:log_cd_z")

key_terms = ['predator_probability', 'distance', 'log_beta_z', 'log_k_z', 'log_cd_z',
             'predator_probability:log_beta_z', 'predator_probability:log_k_z',
             'predator_probability:log_cd_z',
             'distance:log_beta_z', 'distance:log_k_z', 'distance:log_cd_z']

epoch_models = {}
for epoch_name, dv_col in [('Anticipatory', 'antic_vigor_resid'),
                            ('Reactive',     'react_vigor_resid'),
                            ('Terminal',     'term_vigor_resid')]:
    sub = ed.dropna(subset=[dv_col]).copy()
    if len(sub) < 100:
        print(f"\n{epoch_name}: insufficient data ({len(sub)} rows), skipping")
        continue

    f = formula.format(dv=dv_col)
    m = smf.mixedlm(f, data=sub, groups=sub['subj']).fit(reml=False)
    epoch_models[epoch_name] = m

    print(f"\n{'='*75}")
    print(f"  {epoch_name} epoch (N = {len(sub):,} observations)")
    print(f"{'='*75}")
    print(f"  {'Term':<40} {'Coef':>8} {'SE':>8} {'z':>8} {'p':>10}")
    print(f"  {'-'*78}")
    for t in key_terms:
        if t in m.fe_params.index:
            coef = m.fe_params[t]
            se = m.bse_fe[t]
            z = m.tvalues[t]
            p = m.pvalues[t]
            sig = "***" if p < .001 else ("**" if p < .01 else ("*" if p < .05 else ("." if p < .1 else "")))
            print(f"  {t:<40} {coef:>8.4f} {se:>8.4f} {z:>8.3f} {p:>10.4f} {sig}")

In [ ]:
# ── H5a-c Verdict ──
print("=" * 65)
print("H5a-c VERDICT SUMMARY")
print("=" * 65)

def get_pval(models, epoch, term):
    if epoch in models and term in models[epoch].pvalues.index:
        return models[epoch].pvalues[term], models[epoch].tvalues[term]
    return np.nan, np.nan

# H5a: threat x beta — sig anticipatory, null reactive
p_a, z_a = get_pval(epoch_models, 'Anticipatory', 'predator_probability:log_beta_z')
p_r, z_r = get_pval(epoch_models, 'Reactive', 'predator_probability:log_beta_z')
print(f"\nH5a — threat x beta:")
print(f"  Anticipatory: z = {z_a:.3f}, p = {p_a:.4f}  (expect p < 0.05) {'PASS' if p_a < 0.05 else 'FAIL'}")
print(f"  Reactive:     z = {z_r:.3f}, p = {p_r:.4f}  (expect p > 0.10) {'PASS' if p_r > 0.10 else 'FAIL'}")

# H5b: distance x cd — null anticipatory, sig reactive
p_a2, z_a2 = get_pval(epoch_models, 'Anticipatory', 'distance:log_cd_z')
p_r2, z_r2 = get_pval(epoch_models, 'Reactive', 'distance:log_cd_z')
print(f"\nH5b — distance x c_d:")
print(f"  Anticipatory: z = {z_a2:.3f}, p = {p_a2:.4f}  (expect p > 0.10) {'PASS' if p_a2 > 0.10 else 'FAIL'}")
print(f"  Reactive:     z = {z_r2:.3f}, p = {p_r2:.4f}  (expect p < 0.01) {'PASS' if p_r2 < 0.01 else 'FAIL'}")

# H5c: cd main — null anticipatory, sig terminal
p_a3, z_a3 = get_pval(epoch_models, 'Anticipatory', 'log_cd_z')
p_t3, z_t3 = get_pval(epoch_models, 'Terminal', 'log_cd_z')
print(f"\nH5c — c_d main effect:")
print(f"  Anticipatory: z = {z_a3:.3f}, p = {p_a3:.4f}  (expect p > 0.10) {'PASS' if p_a3 > 0.10 else 'FAIL'}")
print(f"  Terminal:     z = {z_t3:.3f}, p = {p_t3:.4f}  (expect p < 0.001) {'PASS' if p_t3 < 0.001 else 'FAIL'}")

## H5d — Threat independence test

Threat predicts vigor in anticipatory (p < 0.01) but not terminal (p > 0.10).  
Compute r(vigor_resid, threat) per epoch.

In [ ]:
# ── H5d: Threat-vigor correlation per epoch ──
print("H5d — Threat-vigor correlation by epoch")
print("=" * 55)

for epoch, col in [('Anticipatory', 'antic_vigor_resid'),
                   ('Reactive',     'react_vigor_resid'),
                   ('Terminal',     'term_vigor_resid')]:
    sub = ed.dropna(subset=[col])
    r, p = pearsonr(sub['predator_probability'], sub[col])
    print(f"  {epoch:15s}: r = {r:.4f}, p = {p:.2e}, N = {len(sub):,}")

# Specific verdicts
sub_a = ed.dropna(subset=['antic_vigor_resid'])
r_a, p_a = pearsonr(sub_a['predator_probability'], sub_a['antic_vigor_resid'])
sub_t = ed.dropna(subset=['term_vigor_resid'])
if len(sub_t) > 10:
    r_t, p_t = pearsonr(sub_t['predator_probability'], sub_t['term_vigor_resid'])
else:
    r_t, p_t = np.nan, np.nan

print(f"\nVerdict:")
print(f"  Anticipatory: r = {r_a:.4f}, p = {p_a:.2e}  (expect p < 0.01) {'PASS' if p_a < 0.01 else 'FAIL'}")
if not np.isnan(p_t):
    print(f"  Terminal:     r = {r_t:.4f}, p = {p_t:.4f}  (expect p > 0.10) {'PASS' if p_t > 0.10 else 'FAIL'}")
else:
    print(f"  Terminal:     insufficient data")

## H5e — Reactive vigor: attack_trial drives vigor, not stated probability

**Model:** `react_vigor_resid ~ attack_trial + predator_probability + attack_trial:predator_probability + (1 | subj)`  
**Test:** attack_trial significant, interaction null

In [ ]:
# ── H5e: Threat independence in reactive epoch ──
react_df = ed.dropna(subset=['react_vigor_resid']).copy()

model_h5e = smf.mixedlm(
    "react_vigor_resid ~ is_attack + predator_probability + is_attack:predator_probability",
    data=react_df,
    groups=react_df['subj']
).fit(reml=False)

print("H5e — Reactive vigor: attack_trial vs predator_probability")
print("=" * 65)
for t in ['is_attack', 'predator_probability', 'is_attack:predator_probability']:
    if t in model_h5e.fe_params.index:
        z = model_h5e.tvalues[t]
        p = model_h5e.pvalues[t]
        b = model_h5e.fe_params[t]
        sig = "***" if p < .001 else ("**" if p < .01 else ("*" if p < .05 else ""))
        print(f"  {t:<40} beta={b:.4f}, z={z:.3f}, p={p:.4f} {sig}")

z_attack = model_h5e.tvalues.get('is_attack', np.nan)
p_attack = model_h5e.pvalues.get('is_attack', np.nan)
p_inter  = model_h5e.pvalues.get('is_attack:predator_probability', np.nan)

print(f"\nVerdict:")
print(f"  attack_trial: z = {z_attack:.3f}, p = {p_attack:.2e}  (expect significant) {'PASS' if p_attack < 0.05 else 'FAIL'}")
print(f"  interaction:  p = {p_inter:.4f}  (expect null, p > 0.05) {'PASS' if p_inter > 0.05 else 'FAIL'}")

## H5f — Variance compression

Between-subject variance in residualized vigor should be compressed in the reactive epoch relative to the anticipatory epoch (ratio > 3; exploratory: 6.6x).

In [ ]:
# ── H5f: Variance compression ──
# Between-subject SD of mean residualized vigor per epoch
subj_means = {}
for epoch, col in [('Anticipatory', 'antic_vigor_resid'),
                   ('Reactive',     'react_vigor_resid'),
                   ('Terminal',     'term_vigor_resid')]:
    sub = ed.dropna(subset=[col])
    sm_epoch = sub.groupby('subj')[col].mean()
    subj_means[epoch] = sm_epoch

sd_antic = subj_means['Anticipatory'].std()
sd_react = subj_means['Reactive'].std()
sd_term  = subj_means.get('Terminal', pd.Series()).std() if 'Terminal' in subj_means else np.nan
ratio = sd_antic / sd_react if sd_react > 0 else np.nan

print("H5f — Variance Compression")
print("=" * 55)
print(f"  Anticipatory SD = {sd_antic:.4f}")
print(f"  Reactive SD     = {sd_react:.4f}")
if not np.isnan(sd_term):
    print(f"  Terminal SD     = {sd_term:.4f}")
print(f"\n  Ratio (antic/react) = {ratio:.1f}x")
print(f"\n  Verdict: {'PASS' if ratio > 3 else 'FAIL'} (threshold: ratio > 3)")

# Visualization
fig, ax = plt.subplots(figsize=(5, 4))
epochs = ['Anticipatory', 'Reactive']
sds = [sd_antic, sd_react]
if not np.isnan(sd_term):
    epochs.append('Terminal')
    sds.append(sd_term)
colors = ['steelblue', 'indianred', 'seagreen'][:len(epochs)]
ax.bar(epochs, sds, color=colors, edgecolor='white')
ax.set_ylabel('Between-Subject SD of Mean Vigor Residual')
ax.set_title(f'H5f: Variance Compression ({ratio:.1f}x)')
for i, (e, s) in enumerate(zip(epochs, sds)):
    ax.text(i, s + 0.001, f'{s:.4f}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

## Beta tertile x threat figure data (anticipatory vs reactive)

In [ ]:
# ── Beta tertile x threat: anticipatory vs reactive ──
ed_plt = ed.copy()
beta_terciles = pd.qcut(params.set_index('subj')['log_beta_z'], 3, labels=['Low beta', 'Med beta', 'High beta'])
ed_plt = ed_plt.merge(beta_terciles.rename('beta_group').reset_index(), on='subj')

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, (epoch, col) in zip(axes, [('Anticipatory', 'antic_vigor_resid'),
                                     ('Reactive', 'react_vigor_resid')]):
    sub = ed_plt.dropna(subset=[col])
    for i, (grp, color) in enumerate(zip(['Low beta', 'Med beta', 'High beta'],
                                          ['#2196F3', '#FF9800', '#F44336'])):
        g = sub[sub['beta_group'] == grp]
        means = g.groupby('T_round')[col].agg(['mean', 'sem'])
        offset = (i - 1) * 0.03
        ax.errorbar(means.index + offset, means['mean'], yerr=means['sem'] * 1.96,
                    marker='o', label=grp, color=color, capsize=3, lw=1.5)
    ax.set_xlabel('Threat Probability')
    ax.set_ylabel('Residualized Vigor')
    ax.set_title(epoch)
    ax.axhline(0, color='gray', ls='--', alpha=0.4)
    ax.set_xticks([0.1, 0.5, 0.9])
    ax.legend(fontsize=8)

plt.suptitle('Beta Tertile x Threat: Anticipatory vs Reactive', y=1.02)
plt.tight_layout()
plt.show()

## c_d tertile x distance figure data

In [ ]:
# ── c_d tertile x distance: anticipatory vs reactive ──
cd_terciles = pd.qcut(params.set_index('subj')['log_cd_z'], 3, labels=['Low c_d', 'Med c_d', 'High c_d'])
ed_plt2 = ed.copy()
ed_plt2 = ed_plt2.merge(cd_terciles.rename('cd_group').reset_index(), on='subj')

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, (epoch, col) in zip(axes, [('Anticipatory', 'antic_vigor_resid'),
                                     ('Reactive', 'react_vigor_resid')]):
    sub = ed_plt2.dropna(subset=[col])
    for i, (grp, color) in enumerate(zip(['Low c_d', 'Med c_d', 'High c_d'],
                                          ['#4CAF50', '#FF9800', '#9C27B0'])):
        g = sub[sub['cd_group'] == grp]
        means = g.groupby('distance')[col].agg(['mean', 'sem'])
        offset = (i - 1) * 0.08
        ax.errorbar(means.index + offset, means['mean'], yerr=means['sem'] * 1.96,
                    marker='s', label=grp, color=color, capsize=3, lw=1.5)
    ax.set_xlabel('Distance')
    ax.set_ylabel('Residualized Vigor')
    ax.set_title(epoch)
    ax.axhline(0, color='gray', ls='--', alpha=0.4)
    ax.set_xticks([1, 2, 3])
    ax.legend(fontsize=8)

plt.suptitle('c_d Tertile x Distance: Anticipatory vs Reactive', y=1.02)
plt.tight_layout()
plt.show()

## Handoff Summary Table

In [ ]:
# ── Handoff Summary Table ──
print("=" * 80)
print("H5 HANDOFF SUMMARY — Threat Imminence Gradient")
print("=" * 80)
print()
print(f"{'Test':<45} {'Anticipatory':>14} {'Reactive':>14} {'Terminal':>14}")
print(f"{'-'*87}")

tests = [
    ('H5a: threat x beta',        'predator_probability:log_beta_z'),
    ('H5b: distance x c_d',       'distance:log_cd_z'),
    ('H5c: c_d main effect',      'log_cd_z'),
]

for label, term in tests:
    vals = []
    for epoch in ['Anticipatory', 'Reactive', 'Terminal']:
        if epoch in epoch_models and term in epoch_models[epoch].pvalues.index:
            z = epoch_models[epoch].tvalues[term]
            p = epoch_models[epoch].pvalues[term]
            sig = '***' if p < .001 else ('**' if p < .01 else ('*' if p < .05 else 'ns'))
            vals.append(f"z={z:.2f} {sig}")
        else:
            vals.append("—")
    print(f"  {label:<43} {vals[0]:>14} {vals[1]:>14} {vals[2]:>14}")

print(f"\n  H5d: r(threat, vigor)                      r={r_a:.3f}***       —              {'r='+f'{r_t:.3f} ns' if not np.isnan(r_t) else '—'}")
print(f"  H5e: attack_trial drives reactive          —              z={z_attack:.2f}***      —")
print(f"  H5f: variance compression                  SD={sd_antic:.4f}      SD={sd_react:.4f}     ratio={ratio:.1f}x")
print()
print("*** p<.001, ** p<.01, * p<.05, ns = not significant")

## Summary

| Test | Prediction | Anticipatory | Reactive | Terminal | Verdict |
|------|-----------|-------------|----------|----------|---------|
| H5a (threat x beta) | sig antic, null react | _fill_ | _fill_ | — | _PASS/FAIL_ |
| H5b (distance x c_d) | null antic, sig react | _fill_ | _fill_ | — | _PASS/FAIL_ |
| H5c (c_d main) | null antic, sig terminal | _fill_ | — | _fill_ | _PASS/FAIL_ |
| H5d (threat-vigor r) | sig antic, null terminal | _fill_ | — | _fill_ | _PASS/FAIL_ |
| H5e (attack drives reactive) | attack sig, interaction null | — | _fill_ | — | _PASS/FAIL_ |
| H5f (variance compression) | ratio > 3 | _fill_ | _fill_ | — | _PASS/FAIL_ |

**Interpretation:** If H5a-f all pass, effort-threat integration (governed by beta from the choice model) is confined to the pre-encounter deliberative phase. At predator encounter, a separate reactive defense system (governed by c_d and actual predator presence, not stated probability) takes over. This is the threat imminence gradient — the core claim of the paper.